# 🧠 Medical Chatbot using RAG Architecture

This notebook builds a **Medical Question Answering Chatbot** using:

- LangChain
- Pinecone Vector Database
- HuggingFace Embeddings
- OpenAI LLM

---

## 📌 What is RAG?

RAG (Retrieval Augmented Generation) means:

1. Search relevant information from documents
2. Send that information to LLM
3. Generate accurate answers

Instead of guessing, the model answers using real data.


In [1]:
# ==========================================================
# STEP 1: Basic Environment Check
# ==========================================================

# Print confirmation message
print("Notebook is running successfully")

# Show current working directory
%pwd


Notebook is running successfully


'f:\\AI Projects\\End-to-End-GenAI-RAG-Based-Application-Medical-Chatbot\\notebook_experiment'

In [2]:
import os
os.chdir("../")
%pwd

'f:\\AI Projects\\End-to-End-GenAI-RAG-Based-Application-Medical-Chatbot'

## 📂 Step 2 — Import Required Libraries

We import modules required for:

- Loading PDFs
- Processing documents
- Creating embeddings
- Connecting vector database


In [3]:
# ==========================================================
# STEP 2: Import Libraries
# ==========================================================

import os

# Document loaders
from langchain.document_loaders import PyPDFLoader, DirectoryLoader


f:\AI Projects\End-to-End-GenAI-RAG-Based-Application-Medical-Chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 📄 Step 3 — Load PDF Documents

We load all medical PDFs stored inside the `data/` folder.

Each PDF becomes a **Document object**.


In [4]:
# ==========================================================
# STEP 3: Load PDFs
# ==========================================================

def load_pdf_files(data_path):
    """
    Loads all PDF files from a directory.

    Parameters:
        data_path (str): folder containing PDFs

    Returns:
        list: extracted documents
    """
    
    loader = DirectoryLoader(
        data_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    
    documents = loader.load()
    return documents


# Load documents
extracted_data = load_pdf_files("data")

print("Total documents loaded:", len(extracted_data))


Total documents loaded: 637


## ✂️ Step 4 — Split Documents into Chunks

Large documents are divided into smaller parts.

### Why?

LLMs cannot process very long text efficiently.

Chunking improves:
- search accuracy
- retrieval speed
- answer quality


In [5]:
# ==========================================================
# STEP 4: Text Chunking
# ==========================================================

from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_documents(documents):
    """
    Splits documents into smaller chunks.
    """
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,     # characters per chunk
        chunk_overlap=100   # overlapping characters
    )

    chunks = text_splitter.split_documents(documents)
    return chunks


texts_chunk = split_documents(extracted_data)

print("Number of chunks created:", len(texts_chunk))


Number of chunks created: 4536


## 🔢 Step 5 — Create Embeddings

Embeddings convert text into numbers.

Example:

"What is Acne?"
↓
[0.12, -0.44, 0.91, ...]  (384 numbers)

These numbers help computers understand meaning.


In [6]:
# ==========================================================
# STEP 5: Load Embedding Model
# ==========================================================

from langchain.embeddings import HuggingFaceEmbeddings

def load_embeddings():
    """
    Loads sentence transformer embedding model.
    """
    
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    
    return embeddings


embedding = load_embeddings()

# Test embedding
vector = embedding.embed_query("Hello world")

print("Embedding dimension:", len(vector))


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4736\3135453902.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Embedding dimension: 384


## 🔐 Step 6 — Load API Keys

We use `.env` file to securely store keys.


In [7]:
# ==========================================================
# STEP 6: Load Environment Variables
# ==========================================================

from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


## 🗄 Step 7 — Connect to Pinecone Vector Database

Vector DB stores embeddings and enables semantic search.


In [10]:
# ==========================================================
# STEP 7: Pinecone Connection
# ==========================================================

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-chatbot"

# Create index if it doesn't exist
if index_name not in pc.list_indexes().names():
    
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )


## 📥 Step 8 — Store Embeddings in Vector Database


In [11]:
# ==========================================================
# STEP 8: Upload Documents to Pinecone
# ==========================================================

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

print("Documents stored successfully!")


Documents stored successfully!


## 🔎 Step 9 — Create Retriever

Retriever finds most relevant chunks based on question meaning.


In [12]:
# ==========================================================
# STEP 9: Retriever
# ==========================================================

retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # return top 3 results
)


## 🤖 Step 10 — Load LLM


In [13]:
# ==========================================================
# STEP 10: Load Chat Model
# ==========================================================

from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)




## 🔗 Step 11 — Build RAG Chain


In [14]:
# ==========================================================
# STEP 11: Create RAG Pipeline
# ==========================================================

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate

system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(chatModel, prompt)

rag_chain = create_retrieval_chain(retriever, question_answer_chain)


## 💬 Step 12 — Ask Questions


In [15]:
# ==========================================================
# STEP 12: Query the Chatbot
# ==========================================================

response = rag_chain.invoke({
    "input": "What is Acne?"
})

print(response["answer"])


Acne is a common skin condition that occurs when pores or hair follicles become blocked. This blockage is caused by oil (sebum), dead skin cells, and bacteria, leading to the collection of a waxy material inside the pores. It commonly results in pimples, blackheads, and whiteheads, primarily on the face, chest, shoulders, and back.


In [17]:
response = rag_chain.invoke({"input": "what are key terms of Abdominal ultrasound?"})
print(response["answer"])

Key terms related to abdominal ultrasound include the procedure itself, which detects suspected abnormalities such as abdominal mass, tumors, cysts, abscesses, scar tissue, and abdominal aortic aneurysm. It is often compared to computed tomography scans (CT) and can involve techniques like M-mode and Doppler-enhanced scans. A crucial consideration is the potential injury to a fetus in early development.


In [18]:
response = rag_chain.invoke({"input": "What is Abdominal wall defects?"})
print(response["answer"])

Abdominal wall defects are congenital birth defects where the stomach or intestines protrude outside the baby's abdomen. This occurs when the umbilical opening is too large or develops improperly, preventing the abdominal wall from fully enclosing these organs. During fetal development, the stomach and intestines initially develop outside the abdomen before being enclosed.


# ✅ Final Pipeline Summary

PDF → Chunk → Embedding → Pinecone → Retriever → LLM → Answer

This is a complete **RAG-based AI application**.
